In [13]:
import torch
import pandas as pd
from pathlib import Path

loss_type = 'hinge'  # 'logistic' or 'hinge'
# branch = "synthetic"


synthetic = False

if synthetic:
    emb_dir = Path("../embeddings_synthetic_pca_64")
    vec_dir = Path(f"pca_64/synthetic_last_pca/{loss_type}")

    proj_dir = Path(f"projection_last_pca_{loss_type}_within_synthetic")

else: 
    emb_dir = Path("../embeddings_pca_64")
    vec_dir = Path(f"pca_64/last_pca/{loss_type}")

    proj_dir = Path(f"projection_last_pca_{loss_type}_within_real")
proj_dir.mkdir(exist_ok=True)


pt_files = sorted(vec_dir.glob("*.pt"))
records = []

for path in pt_files:
    data = torch.load(path, map_location="cpu")
    records.append(data)

# Build a lookup: (model_short, benchmark, column) -> diffmean tensor [L, D]
dm_map = {}
for r in records:  # records from the diffmean-per-layer cell
    key = (r["model"], r["benchmark"], r["column"])
    dm_map[key] = torch.tensor(r["W"]).clone().detach()  # [L, D]

print(f"Loaded {len(dm_map)} diffmean entries")

pt_files = sorted(emb_dir.glob("*.pt"))
print(f"Found {len(pt_files)} embedding files in {emb_dir}")

for path in pt_files:
    print(f"\n=== Processing {path} ===")
    data = torch.load(path, map_location="cpu")

    emb_all = data["embeddings"]          # [N, L, D]
    model_name = data["model_name"]       # full HF name
    benchmark = data["benchmark"]         # e.g. "RQ"
    column = data["column"]               # e.g. "question_with_context"
    model_short = model_name.split("/")[-1]

    # load the full CSV for this benchmark
    csv_path = f"../{benchmark}.csv"
    df = pd.read_csv(csv_path)

    print(f"benchmark={benchmark}, column={column}, model={model_name}")
    print(f"embeddings shape: {emb_all.shape}, df rows: {len(df)}")

    key = (model_short, benchmark, column)
    if key not in dm_map:
        continue
        raise KeyError(f"No diffmean found for {key}")

    for second_key in dm_map:
        if second_key == key:
            continue
        
        model_dm, benchmark_dm, column_dm = second_key
        
        if model_dm != model_short:
            continue
        print(f"Diffmean key: model={model_dm}, benchmark={benchmark_dm}, column={column_dm}")
        diffmean = dm_map[key]    # [L, D]
        
        
        # Use id to align DF rows with embedding rows
        ids = df["id"].to_numpy()
        labels = df["binary_label"].to_numpy()
        datasets = df["dataset"].tolist()     # keep as list of strings

        ids_t = torch.as_tensor(ids, dtype=torch.long)
        assert ids_t.max().item() < emb_all.shape[0], (
            f"Max id {ids_t.max().item()} >= num embeddings {emb_all.shape[0]}"
        )

        # X: [num_rows, L, D]
        X = emb_all[ids_t]

        num_rows, num_layers, embed_dim = X.shape
        assert diffmean.shape == (num_layers, embed_dim), (
            f"Shape mismatch: X {X.shape}, diffmean {diffmean.shape}"
        )

        print(f"num_rows={num_rows}, num_layers={num_layers}, embed_dim={embed_dim}")

        eps = 1e-8

        # Normalize ONLY the diffmean vectors: d_hat_l = d_l / ||d_l||
        dm_unit = diffmean / (diffmean.norm(dim=-1, keepdim=True) + eps)   # [L, D]

        # Projection: x_{i,l} ⋅ d_hat_l
        # ( [num_rows,L,D] * [1,L,D] ) -> [num_rows,L,D] -> sum over D -> [num_rows,L]
        proj = (X * dm_unit.unsqueeze(0)).sum(dim=-1)   # [num_rows, L]

        print(f"projection shape: {proj.shape}")  # [num_rows, L]

        out = {
            "projection": proj,                          # [num_rows, L]
            "id": torch.as_tensor(ids, dtype=torch.long),
            "binary_label": torch.as_tensor(labels),     # same order as rows in df
            "dataset": datasets,                         # list of split names per row
            "benchmark": benchmark,
            "column": column,
            "model_name": model_name,
            "benchmark_primary": benchmark_dm,
            "column_primary": column_dm,
        }

        out_path = proj_dir / f"{benchmark}_{column}_TO_{benchmark_dm}_{column_dm}_{model_short}_projection.pt"
        torch.save(out, out_path)
        print(f"Saved projections to {out_path}")



/var/folders/1d/x42w8kj57yg4hd8tjgzqc8t40000gn/T/ipykernel_79309/1775205421.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dm_map[key] = torch.tensor(r["W"]).clone().detach()  # [L, D]


Loaded 25 diffmean entries
Found 30 embedding files in ../embeddings_pca_64

=== Processing ../embeddings_pca_64/RQ_question_Llama-3.1-8B-Instruct.pt ===
benchmark=RQ, column=question, model=meta-llama/Llama-3.1-8B-Instruct
embeddings shape: torch.Size([4997, 33, 64]), df rows: 4997
Diffmean key: model=Llama-3.1-8B-Instruct, benchmark=RQ, column=question_with_context
num_rows=4997, num_layers=33, embed_dim=64
projection shape: torch.Size([4997, 33])
Saved projections to projection_last_pca_hinge_within_real/RQ_question_TO_RQ_question_with_context_Llama-3.1-8B-Instruct_projection.pt
Diffmean key: model=Llama-3.1-8B-Instruct, benchmark=SRAQ, column=full_turn
num_rows=4997, num_layers=33, embed_dim=64
projection shape: torch.Size([4997, 33])
Saved projections to projection_last_pca_hinge_within_real/RQ_question_TO_SRAQ_full_turn_Llama-3.1-8B-Instruct_projection.pt
Diffmean key: model=Llama-3.1-8B-Instruct, benchmark=SRAQ, column=longer_context
num_rows=4997, num_layers=33, embed_dim=64
pr

# AUROC

In [14]:
import json
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import roc_auc_score  # make sure scikit-learn is installed

def auc_confint_hanley_mcneil(y_true, y_score, alpha=0.05):
    """
    y_true: 0/1 labels
    y_score: continuous scores
    Returns: auc, lower, upper  (95% CI by default)
    """
    auc = roc_auc_score(y_true, y_score)

    y_true = np.asarray(y_true)
    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()

    Q1 = auc / (2.0 - auc)
    Q2 = 2.0 * auc**2 / (1.0 + auc)

    var_auc = (
        auc * (1.0 - auc)
        + (n_pos - 1.0) * (Q1 - auc**2)
        + (n_neg - 1.0) * (Q2 - auc**2)
    ) / (n_pos * n_neg)

    se = np.sqrt(var_auc)
    z = 1.96  # for 95% CI

    lower = auc - z * se
    upper = auc + z * se

    # clamp to [0,1]
    lower = max(0.0, lower)
    upper = min(1.0, upper)

    return float(auc), float(lower), float(upper)

In [19]:
synthetic = False
metric = 'hinge'  # 'logistic' or 'hinge'


if synthetic:
    proj_dir = Path(f"projection_last_pca_{metric}_within_synthetic")
    out_path = Path(f"auroc_project_within_synthetic_{metric}.jsonl")
else:
    proj_dir = Path(f"projection_last_pca_{metric}_within_real")
    out_path = Path(f"auroc_project_within_real_{metric}.jsonl")

pt_files = sorted(proj_dir.glob("*.pt"))
print(f"Found {len(pt_files)} projection files in {proj_dir}")

with open(out_path, "w", encoding="utf-8") as out_f:
    for path in pt_files:
        print(f"\n=== Processing {path} ===")
        data = torch.load(path, map_location="cpu")

        proj = data["projection"]            # [N, L]
        labels = data["binary_label"]        # tensor [N]
        datasets = data["dataset"]           # list of length N
        benchmark = data["benchmark"]
        column = data["column"]
        primary_benchmark = data["benchmark_primary"]
        primary_column = data["column_primary"]
        model_name = data["model_name"]
        model_short = model_name.split("/")[-1]

        proj_np = proj.numpy()               # [N, L]
        y_np = labels.numpy()                # [N]

        N, num_layers = proj_np.shape
        print(f"N={N}, num_layers={num_layers}")

        # preserve split order as they first appear in datasets
        seen = set()
        splits = []
        for s in datasets:
            if s not in seen:
                seen.add(s)
                splits.append(s)

        record = {
            "benchmark": benchmark,
            "column": column,
            "model": model_short,
            "primary_benchmark": primary_benchmark,
            "primary_column": primary_column,
        }
        test_split = "test"
        print(f"Computing AUROC on split '{test_split}'")
        indices = [i for i, s in enumerate(datasets) if s == test_split]
        y_split = y_np[indices]
        X_split = proj_np[indices]  

        aurocs = []
        aurocs_ci = []
        for layer_idx in range(num_layers):
            scores = X_split[:, layer_idx]
            try:
                auc, lo, hi = auc_confint_hanley_mcneil(y_split, scores)
            except ValueError:
                # happens if y_split is all 0s or all 1s
                auc, lo, hi = None, None, None
            aurocs.append(auc)
            aurocs_ci.append([lo, hi])

        # e.g. record["train"] = [auroc_layer0, ...]
        #      record["train_ci"] = [[lo0, hi0], [lo1, hi1], ...]
        record['auroc'] = aurocs
        record['auroc_ci'] = aurocs_ci

        out_f.write(json.dumps(record) + "\n")
        out_f.flush()
        print(f"Wrote AUROC record for {benchmark}/{column} ({model_short})")

print(f"\nAll done. AUROCs saved to {out_path}")


Found 100 projection files in projection_last_pca_hinge_within_real

=== Processing projection_last_pca_hinge_within_real/RQ_question_TO_RQ_question_with_context_Llama-3.1-8B-Instruct_projection.pt ===
N=4997, num_layers=33
Computing AUROC on split 'test'
Wrote AUROC record for RQ/question (Llama-3.1-8B-Instruct)

=== Processing projection_last_pca_hinge_within_real/RQ_question_TO_RQ_question_with_context_Llama-3.3-70B-Instruct_projection.pt ===
N=4997, num_layers=81
Computing AUROC on split 'test'
Wrote AUROC record for RQ/question (Llama-3.3-70B-Instruct)

=== Processing projection_last_pca_hinge_within_real/RQ_question_TO_RQ_question_with_context_Qwen3-32B_projection.pt ===
N=4997, num_layers=65
Computing AUROC on split 'test'
Wrote AUROC record for RQ/question (Qwen3-32B)

=== Processing projection_last_pca_hinge_within_real/RQ_question_TO_RQ_question_with_context_Qwen3-8B_projection.pt ===
N=4997, num_layers=37
Computing AUROC on split 'test'
Wrote AUROC record for RQ/question (Qw